# HRS Integrated-W Association Test — Study 1 / Exp4 (Preregistered 2026-08-25, First Execution)

Standalone notebook implementing `Beyond-The-Mean/preregistration/PDP_HRS_IntegratedW_Association_AnalysisPlan_2026-08-25.md`,
Sections 3-9, on Study 1 / Exp4 -- a test that was locked but never actually run anywhere findable (see that
plan's Section 14 addendum, added 2026-09-01 after a search of Drive/source-repo/paper-repo turned
up nothing implementing it). Extracted out of the much larger `Variance_Narrowing_executed.ipynb`
(Part 19) into its own file so it can run independently, without the other ~150 cells of unrelated
exploratory work.

**D_H** (per participant): mean(H_RS,Subject − H_RS,PCS) across that participant's eligible blocks.

**A_W** (per participant): mean over W in {3,5,10} of max(0, −Z_W), where Z_W standardizes that
participant's own windowed-r variance at W against a Baseline reference distribution matched to
their session-count structure (session-level Baseline resampling -- see MEMORY.md / this project's
2026-09-03 correction to the earlier participant-level Baseline resampling bug).

**Primary test** (Section 8): Spearman(D_H, A_W) across the full eligible Human population.
**Secondary** (Section 9): Human5+ only -- does not redefine the primary result.
**Null** (Section 6): matched Baseline pseudo-groups, reproducing each real participant's
session-count structure, giving an empirical null for the Spearman correlation itself.

One computational simplification from the plan's literal wording, stated explicitly: the
per-session-count Baseline calibration (mean/SD used to compute each participant's Z_W) is computed
ONCE per distinct session count and reused for both the real Human group and every pseudo-group draw
in the null loop, rather than being freshly resampled inside each of the 1000+ null repetitions.
This is a fixed reference distribution indexed by session-count structure, not a per-iteration
resample -- it makes the difference between a computation that finishes and one that does not, and
does not build in any advantage for the Human group specifically (the same fixed reference is used
to score pseudo-participants too).

**Data**: downloads `Frozen_Blocks_2026-02-10_195735.csv` / `Frozen_Sessions_2026-02-10_195735.csv`
from the `Beyond-The-Mean` GitHub repo's `data/` folder if not found locally (same fallback pattern
as that repo's own Git-variant notebooks) -- these are the same two files Exp4 loading has always
used, part of the citation-grade 5-file manifest.

In [17]:
# ─────────────────────────────────────────────────────────────────────────
# GLOBAL BOOTSTRAP/PERMUTATION ITERATION CONTROL
# Lower this ONE number for a faster, noisier pass on a free-tier Colab CPU; 10000 is the "real"
# setting. Everything below (calibration draws, null-group repetitions) scales off it.
# ─────────────────────────────────────────────────────────────────────────
N_BOOT_DEFAULT = 10000   # try 500-1000 for a quick draft pass

In [18]:
import numpy as np
import pandas as pd
from scipy import stats
import ast, warnings
from pathlib import Path
import urllib.request
warnings.filterwarnings('ignore')

DOWNLOAD_BASE = "https://raw.githubusercontent.com/catboxer/Beyond-The-Mean/main/data"

def locate_required_inputs(filenames, search_root=None, download_base=DOWNLOAD_BASE):
    """Find one unambiguous local copy of every required file, downloading from the Beyond-The-Mean
    repo's data/ folder on GitHub if no local copy is found. No recursive filesystem fallback
    (deliberately -- see Beyond-The-Mean's Git-variant notebooks for why: a broad rglob() from root
    can sweep in unrelated stale duplicate files from a mounted Drive)."""
    root = Path.cwd() if search_root is None else Path(search_root)
    data_dir = root / "data"
    resolved = {}
    problems = []
    for label, filename_pattern in filenames.items():
        candidates = []
        for parent in (root, data_dir):
            if parent.is_dir():
                candidates.extend(path.resolve() for path in parent.glob(filename_pattern) if path.is_file())
        candidates = sorted(set(candidates))
        if not candidates and download_base:
            exact_name = filename_pattern.replace("*", "")
            data_dir.mkdir(parents=True, exist_ok=True)
            dest = data_dir / exact_name
            url = f"{download_base}/{exact_name}"
            print(f"  {label}: not found locally -- downloading {url}")
            try:
                urllib.request.urlretrieve(url, dest)
                candidates = [dest.resolve()]
            except Exception as exc:
                problems.append(f"MISSING: {filename_pattern} (download failed: {exc})")
                continue
        if len(candidates) == 1:
            resolved[label] = str(candidates[0])
        elif len(candidates) == 0:
            problems.append(f"MISSING: {filename_pattern}")
        else:
            locations = ", ".join(str(path) for path in candidates)
            problems.append(f"DUPLICATED accepted input for {label}: {locations}")
    if problems:
        expected = "\n".join(f"  - {pattern}" for pattern in filenames.values())
        details = "\n".join(problems)
        raise FileNotFoundError(
            "Required-input file check failed.\n"
            f"{details}\n\n"
            "Place exactly one copy of each required file in the notebook's working "
            "folder or its data/ subfolder:\n"
            f"{expected}\n"
            f"Current search root: {root.resolve()}"
        )
    return resolved

def hurstApprox(bits):
    """Single-scale R/S Hurst exponent. Exact port of the JS hurstApprox used in the experiment."""
    if bits is None:
        return np.nan
    bits = np.asarray(bits, dtype=int)
    n = bits.size
    if n < 20:
        return np.nan
    x = np.where(bits != 0, 1.0, -1.0)
    mean = float(x.sum() / n)
    y = 0.0; minY = 0.0; maxY = 0.0; s2 = 0.0
    for i in range(n):
        d = x[i] - mean
        y += d
        if y < minY: minY = y
        if y > maxY: maxY = y
        s2 += d * d
    R = maxY - minY
    S = np.sqrt(s2 / n) if s2 > 0 else 1.0
    if not np.isfinite(S) or S == 0: S = 1.0
    ratio = (R / S) if (R / S) != 0 else 1.0
    if not np.isfinite(ratio) or ratio == 0: ratio = 1.0
    H = np.log(ratio) / np.log(n)
    if not np.isfinite(H): H = 0.5
    return float(max(0.0, min(1.0, H)))

def parse_literal(s):
    try:
        return ast.literal_eval(s)
    except Exception:
        return None

def windowed_r_tagged(df_in, W, block_idx_col='block_idx'):
    rows = []
    for sid, g in df_in.groupby('session_id'):
        g = g.sort_values(block_idx_col)
        hs = g['hurst_subj_rough'].values; hp = g['hurst_demon_rough'].values
        pid = g['participant_id'].iloc[0]
        n_windows = len(hs)//W
        for i in range(n_windows):
            a=hs[i*W:(i+1)*W]; b=hp[i*W:(i+1)*W]
            if np.std(a)>0 and np.std(b)>0:
                rows.append({'participant_id': pid, 'session_id': sid, 'r': np.corrcoef(a,b)[0,1]})
    return pd.DataFrame(rows)

print("Setup complete: locate_required_inputs, hurstApprox, parse_literal, windowed_r_tagged defined.")

Setup complete: locate_required_inputs, hurstApprox, parse_literal, windowed_r_tagged defined.


In [19]:
# ── Exp4 data ────────────────────────────────────────────────────────────
EXP4_REQUIRED_INPUTS = {
    "blocks": "Frozen_Blocks_2026-02-10_195735.csv",
    "sessions": "Frozen_Sessions_2026-02-10_195735.csv",
}
EXP4_INPUT_PATHS = locate_required_inputs(EXP4_REQUIRED_INPUTS, search_root="/content/drive/MyDrive/QART_Project")
FROZEN_BLOCKS_PATH   = EXP4_INPUT_PATHS["blocks"]
FROZEN_SESSIONS_PATH = EXP4_INPUT_PATHS["sessions"]
print(f"Exp4 blocks:   {FROZEN_BLOCKS_PATH}")
print(f"Exp4 sessions: {FROZEN_SESSIONS_PATH}")

df_blocks = pd.read_csv(FROZEN_BLOCKS_PATH)
df_sessions = pd.read_csv(FROZEN_SESSIONS_PATH)
df4 = df_blocks.copy()
df4['parsed'] = df4['trial_data'].apply(parse_literal)
df4['subject_bits'] = df4['parsed'].apply(lambda x: x.get('subject_bits') if x else None)
df4['demon_bits']   = df4['parsed'].apply(lambda x: x.get('demon_bits') if x else None)
df4 = df4.rename(columns={'sessionId': 'session_id'})
df_sessions4 = df_sessions.rename(columns={'sessionId': 'session_id', 'agent_class': 'condition'})
df4 = df4.merge(df_sessions4[['session_id', 'condition', 'participant_id', 'createdAt']], on='session_id', how='left')
df4 = df4.dropna(subset=['subject_bits', 'demon_bits', 'condition']).copy()
bps4 = df4.groupby('session_id').size()
df4 = df4[df4['session_id'].isin(bps4[bps4 >= 25].index)].copy()
vsc4 = df4.groupby(['participant_id', 'condition'])['session_id'].nunique().reset_index()
vsc4.columns = ['participant_id', 'condition', 'session_count']
df4 = df4.merge(vsc4, on=['participant_id', 'condition'], how='left')
print("Computing Exp4 hurstApprox...", end="", flush=True)
df4['hurst_subj_rough'] = df4['subject_bits'].apply(hurstApprox)
df4['hurst_demon_rough'] = df4['demon_bits'].apply(hurstApprox)
print(" done")

human_all_4 = df4[df4['condition'] == 'human'].copy()
human5_4    = human_all_4[human_all_4['session_count'] >= 5].copy()
baseline_4  = df4[df4['condition'] == 'baseline'].copy()

print(f"Exp4: Human(all)={human_all_4['participant_id'].nunique()} participants "
      f"({len(human_all_4)} blocks), Human5+={human5_4['participant_id'].nunique()} "
      f"({len(human5_4)} blocks), Baseline={baseline_4['participant_id'].nunique()} "
      f"({len(baseline_4)} blocks)")

Exp4 blocks:   /content/drive/MyDrive/QART_Project/data/Frozen_Blocks_2026-02-10_195735.csv
Exp4 sessions: /content/drive/MyDrive/QART_Project/data/Frozen_Sessions_2026-02-10_195735.csv
Computing Exp4 hurstApprox... done
Exp4: Human(all)=121 participants (4737 blocks), Human5+=3 (840 blocks), Baseline=3 (3090 blocks)


## D_H — Section 3.1

In [20]:
def compute_D_H(df_in):
    d = df_in.copy()
    d['h_diff'] = d['hurst_subj_rough'] - d['hurst_demon_rough']
    return d.groupby('participant_id')['h_diff'].mean()

D_H_human_all_4 = compute_D_H(human_all_4)
D_H_human5_4 = compute_D_H(human5_4)
print(f"D_H computed: Human(all)={len(D_H_human_all_4)} participants, Human5+={len(D_H_human5_4)} participants")
print(f"Human(all) D_H: mean={D_H_human_all_4.mean():+.5f}  median={D_H_human_all_4.median():+.5f}")
print(f"Human5+    D_H: mean={D_H_human5_4.mean():+.5f}  median={D_H_human5_4.median():+.5f}")

D_H computed: Human(all)=121 participants, Human5+=3 participants
Human(all) D_H: mean=+0.00034  median=+0.00137
Human5+    D_H: mean=+0.00644  median=+0.00803


## A_W and its Baseline reference distribution — Sections 3.2 and 6

In [21]:
W_VALUES_HRS = [3, 5, 10]
N_CALIB_HRS = max(200, N_BOOT_DEFAULT // 20)   # draws used to build each session-count's Baseline reference distribution

baseline_pools_hrs = {
    W: {sid: g['r'].values for sid, g in windowed_r_tagged(baseline_4, W).groupby('session_id')}
    for W in W_VALUES_HRS
}
baseline_sids_hrs = {W: list(baseline_pools_hrs[W].keys()) for W in W_VALUES_HRS}

_calib_cache = {}

def get_calibration(n_sess, rng):
    """Fixed Baseline reference (mean, sd) of windowed-r variance at each W, for a given session
    count -- computed once per distinct n_sess and cached (see notebook intro: not per-iteration)."""
    if n_sess in _calib_cache:
        return _calib_cache[n_sess]
    out = {}
    for W in W_VALUES_HRS:
        sids = baseline_sids_hrs[W]
        draws = np.empty(N_CALIB_HRS)
        for i in range(N_CALIB_HRS):
            sampled = rng.choice(sids, size=n_sess, replace=True)
            pooled = np.concatenate([baseline_pools_hrs[W][s] for s in sampled])
            draws[i] = pooled.var(ddof=1) if len(pooled) >= 2 else np.nan
        out[W] = (np.nanmean(draws), np.nanstd(draws, ddof=1))
    _calib_cache[n_sess] = out
    return out

print(f"Calibration cache ready. W set: {W_VALUES_HRS}, N_CALIB_HRS={N_CALIB_HRS} draws per (session-count, W).")

Calibration cache ready. W set: [3, 5, 10], N_CALIB_HRS=500 draws per (session-count, W).


In [22]:
def compute_A_W(human_df, rng):
    sess_counts = human_df.groupby('participant_id')['session_id'].nunique()
    results = {}
    for pid, n_sess in sess_counts.items():
        pdf = human_df[human_df['participant_id'] == pid]
        calib = get_calibration(n_sess, rng)
        z_ws = []
        valid = True
        for W in W_VALUES_HRS:
            tagged_p = windowed_r_tagged(pdf, W)
            if len(tagged_p) < 2:
                valid = False
                break
            obs_var = tagged_p['r'].values.var(ddof=1)
            mean_b, sd_b = calib[W]
            if sd_b == 0 or np.isnan(sd_b):
                valid = False
                break
            z_ws.append((obs_var - mean_b) / sd_b)
        if valid:
            results[pid] = np.mean([max(0.0, -z) for z in z_ws])
    return pd.Series(results, dtype=float)

rng_calib = np.random.default_rng(20260908)
A_W_human_all_4 = compute_A_W(human_all_4, rng_calib)
A_W_human5_4 = compute_A_W(human5_4, rng_calib)
print(f"A_W computed: Human(all)={len(A_W_human_all_4)}/{human_all_4['participant_id'].nunique()} participants scored "
      f"(some drop for <2 windows at some W), Human5+={len(A_W_human5_4)}/{human5_4['participant_id'].nunique()}")
print(f"Human(all) A_W: mean={A_W_human_all_4.mean():.5f}  median={A_W_human_all_4.median():.5f}")
print(f"Human5+    A_W: mean={A_W_human5_4.mean():.5f}  median={A_W_human5_4.median():.5f}  "
      f"(bigger mean here than Human(all) would be consistent with the PI's observation that 5+ shows more narrowing)")

A_W computed: Human(all)=121/121 participants scored (some drop for <2 windows at some W), Human5+=3/3
Human(all) A_W: mean=0.39179  median=0.32451
Human5+    A_W: mean=1.18541  median=1.13733  (bigger mean here than Human(all) would be consistent with the PI's observation that 5+ shows more narrowing)


## Primary test — Human(all), Sections 4, 6, 8

In [10]:
N_REPS_HRS = max(200, N_BOOT_DEFAULT // 20)   # matched Baseline pseudo-group repetitions (plan's target: 5000; scaled here)

def spearman_association(D_H_series, A_W_series):
    common = D_H_series.index.intersection(A_W_series.index)
    d = D_H_series.loc[common].values
    a = A_W_series.loc[common].values
    rho, _ = stats.spearmanr(d, a)
    return rho, len(common), common

def null_pseudo_group_association(human_df, rng):
    """One matched Baseline pseudo-group draw: for each real participant, resample that many
    Baseline sessions (matching their session-count structure), compute pseudo D_B and A_W,B the
    same way real D_H/A_W are computed, then return the pseudo-group's own Spearman(D_B, A_W,B)."""
    sess_counts = human_df.groupby('participant_id')['session_id'].nunique()
    d_b, a_wb = [], []
    for pid, n_sess in sess_counts.items():
        sampled_sids = rng.choice(list(baseline_4['session_id'].unique()), size=n_sess, replace=True)
        pseudo_df = baseline_4[baseline_4['session_id'].isin(sampled_sids)]
        if pseudo_df.empty:
            continue
        d_b.append((pseudo_df['hurst_subj_rough'] - pseudo_df['hurst_demon_rough']).mean())
        calib = get_calibration(n_sess, rng)
        z_ws = []
        valid = True
        for W in W_VALUES_HRS:
            tagged_p = windowed_r_tagged(pseudo_df, W)
            if len(tagged_p) < 2:
                valid = False
                break
            obs_var = tagged_p['r'].values.var(ddof=1)
            mean_b, sd_b_ = calib[W]
            if sd_b_ == 0 or np.isnan(sd_b_):
                valid = False
                break
            z_ws.append((obs_var - mean_b) / sd_b_)
        a_wb.append(np.mean([max(0.0, -z) for z in z_ws]) if valid else np.nan)
    d_b = np.array(d_b); a_wb = np.array(a_wb)
    mask = ~np.isnan(a_wb)
    if mask.sum() < min(5, len(sess_counts)):
        return np.nan
    rho, _ = stats.spearmanr(d_b[mask], a_wb[mask])
    return rho

print('='*100); print('PRIMARY -- Exp4 Human(all), Spearman(D_H, A_W) vs Baseline-matched pseudo-group null'); print('='*100)
obs_rho_all, n_all, _ = spearman_association(D_H_human_all_4, A_W_human_all_4)
print(f"Observed Spearman rho = {obs_rho_all:+.4f}  (N={n_all} participants)")

rng_null = np.random.default_rng(20260908)
null_rhos_all = np.array([null_pseudo_group_association(human_all_4, rng_null) for _ in range(N_REPS_HRS)])
null_rhos_all = null_rhos_all[~np.isnan(null_rhos_all)]
one_sided_p = (null_rhos_all >= obs_rho_all).mean()
print(f"Baseline-matched null: N_reps={len(null_rhos_all)} (target {N_REPS_HRS}), "
      f"null rho mean={null_rhos_all.mean():+.4f}, SD={null_rhos_all.std(ddof=1):.4f}")
print(f"One-sided empirical p (fraction of null draws >= observed, positive-direction hypothesis per Section 4) = {one_sided_p:.4f}")

PRIMARY -- Exp4 Human(all), Spearman(D_H, A_W) vs Baseline-matched pseudo-group null
Observed Spearman rho = +0.0182  (N=121 participants)
Baseline-matched null: N_reps=500 (target 500), null rho mean=+0.0910, SD=0.0929
One-sided empirical p (fraction of null draws >= observed, positive-direction hypothesis per Section 4) = 0.7920


## Human5+ sensitivity subgroup — Section 9 (primary test's own hierarchy; does not redefine the primary result above)

In [11]:
print('='*100); print('SECONDARY -- Exp4 Human5+, Spearman(D_H, A_W) vs Baseline-matched pseudo-group null'); print('='*100)
obs_rho_5, n_5, _ = spearman_association(D_H_human5_4, A_W_human5_4)
print(f"Observed Spearman rho = {obs_rho_5:+.4f}  (N={n_5} participants)")

rng_null5 = np.random.default_rng(20260909)
null_rhos_5 = np.array([null_pseudo_group_association(human5_4, rng_null5) for _ in range(N_REPS_HRS)])
null_rhos_5 = null_rhos_5[~np.isnan(null_rhos_5)]
one_sided_p_5 = (null_rhos_5 >= obs_rho_5).mean()
print(f"Baseline-matched null: N_reps={len(null_rhos_5)} (target {N_REPS_HRS}), "
      f"null rho mean={null_rhos_5.mean():+.4f}, SD={null_rhos_5.std(ddof=1):.4f}")
print(f"One-sided empirical p (fraction of null draws >= observed) = {one_sided_p_5:.4f}")
print("\n(Section 9: this subgroup result does not redefine the primary result above and is not "
      "used to rescue a null full-population finding, or vice versa.)")

SECONDARY -- Exp4 Human5+, Spearman(D_H, A_W) vs Baseline-matched pseudo-group null
Observed Spearman rho = -0.5000  (N=3 participants)
Baseline-matched null: N_reps=497 (target 500), null rho mean=+0.0192, SD=0.7122
One-sided empirical p (fraction of null draws >= observed) = 0.8229

(Section 9: this subgroup result does not redefine the primary result above and is not used to rescue a null full-population finding, or vice versa.)


## Primary test summary — reading this once run

Fill in after running. Report the observed Spearman rho and one-sided empirical p for both
Human(all) (primary) and Human5+ (secondary), per Section 10's four predefined patterns: **A**
(holds in both, beyond Baseline expectation), **B** (present in one only -- not directly testable
here since only Exp4 is in this notebook; exp5-prescreen would need this same test run separately),
**C** (present but no better than Baseline's own mechanical association -- i.e. p not small), or
**D** (no clear association). Per Section 12, this stays a follow-up to exploratory findings, not an
independent replication -- the candidate variables were identified by exploring these same
datasets.

## Secondary test — within-participant, session-level (added for the OSF registration)

Registered as a distinct, prospectively specified test alongside the primary one above (see
`OSF_Prereg_HRS_IntegratedW_Association_DRAFT.md`). Asks a different question from the primary
test: within a participant's own sessions, is a session that's higher than expected on D_H,
relative to that participant's own average and trend, also higher than expected on A_W?

Restricted to participants with 3 or more eligible sessions (a session is eligible here only if
`A_W_session` is defined, i.e. it has 2 or more valid windows at every W in {3, 5, 10}; a
participant with fewer than 3 such sessions contributes nothing once order-detrended, since a
2-point linear fit always has zero residual). Evaluated against two nulls: a within-participant
permutation null and a Baseline-mechanical null.

In [12]:
def build_session_table(human_df, rng):
    """One row per eligible session: participant_id, session_id, createdAt, D_H_session,
    A_W_session. A session is dropped if A_W_session is undefined (fewer than 2 valid windows at
    some W in W_VALUES_HRS), using the n=1 Baseline calibration (Variables: A_W_session)."""
    calib_n1 = get_calibration(1, rng)
    rows = []
    for sid, g in human_df.groupby('session_id'):
        pid = g['participant_id'].iloc[0]
        created = g['createdAt'].iloc[0]
        h_diff = (g['hurst_subj_rough'] - g['hurst_demon_rough']).mean()
        z_ws = []
        valid = True
        for W in W_VALUES_HRS:
            tagged = windowed_r_tagged(g, W)
            if len(tagged) < 2:
                valid = False
                break
            obs_var = tagged['r'].values.var(ddof=1)
            mean_b, sd_b = calib_n1[W]
            if sd_b == 0 or np.isnan(sd_b):
                valid = False
                break
            z_ws.append((obs_var - mean_b) / sd_b)
        if not valid:
            continue
        a_w = np.mean([max(0.0, -z) for z in z_ws])
        rows.append({'participant_id': pid, 'session_id': sid, 'createdAt': created,
                      'D_H_session': h_diff, 'A_W_session': a_w})
    return pd.DataFrame(rows)

rng_sess = np.random.default_rng(20260910)
session_table_4 = build_session_table(human_all_4, rng_sess)

sess_counts_sec = session_table_4.groupby('participant_id')['session_id'].nunique()
eligible_pids_sec = sess_counts_sec[sess_counts_sec >= 3].index
session_table_4_eligible = session_table_4[session_table_4['participant_id'].isin(eligible_pids_sec)].copy()

print(f"Secondary analysis eligibility: {len(eligible_pids_sec)}/{session_table_4['participant_id'].nunique()} "
      f"participants have 3 or more sessions with a defined A_W_session "
      f"({len(session_table_4_eligible)} sessions total pooled).")

Secondary analysis eligibility: 6/121 participants have 3 or more sessions with a defined A_W_session (39 sessions total pooled).


In [13]:
def detrend_group(sub):
    """Order-detrend one participant's sessions: sort by createdAt, regress D_H_session and
    A_W_session on session index (1..n), replace each with its residual (dD_H, dA_W)."""
    sub = sub.sort_values('createdAt').copy()
    n = len(sub)
    idx = np.arange(1, n + 1, dtype=float)
    for col, out_col in [('D_H_session', 'dD_H'), ('A_W_session', 'dA_W')]:
        y = sub[col].values
        slope, intercept, *_ = stats.linregress(idx, y)
        sub[out_col] = y - (intercept + slope * idx)
    return sub

session_table_4_eligible = (session_table_4_eligible
    .groupby('participant_id', group_keys=False)
    .apply(detrend_group))

obs_rho_sec, _ = stats.spearmanr(session_table_4_eligible['dD_H'], session_table_4_eligible['dA_W'])
obs_pearson_sec, _ = stats.pearsonr(session_table_4_eligible['dD_H'], session_table_4_eligible['dA_W'])
print(f"Secondary observed Spearman rho = {obs_rho_sec:+.4f}, N sessions = {len(session_table_4_eligible)}, "
      f"N participants = {session_table_4_eligible['participant_id'].nunique()}")
print(f"Secondary Pearson r (secondary/descriptive only) = {obs_pearson_sec:+.4f}")

Secondary observed Spearman rho = +0.0340, N sessions = 39, N participants = 6
Secondary Pearson r (secondary/descriptive only) = -0.0250


### Two nulls for the secondary test

(a) Within-participant permutation null: shuffle which session's A_W_session pairs with which
session's D_H_session, within each participant only, then recompute order-detrending fresh on
the shuffled arrangement. Rules out coincidental pairing.

(b) Baseline-mechanical null: mirrors the primary test's Baseline-matched pseudo-group null
(Section 6), applied per session. For each eligible participant, resample that many Baseline
sessions to build one pseudo-participant, using the resample-draw order as its session index
(Baseline sessions have no real chronological order); apply the same order-detrending. Rules out
a shared computational artifact.

In [14]:
def null_permutation_secondary(sess_table, rng):
    """One within-participant permutation draw: within each eligible participant, keep
    D_H_session in its real session order but shuffle A_W_session's assignment to that order,
    then recompute order-detrending fresh on the shuffled arrangement. Returns the pooled
    Spearman across all eligible sessions for this one draw."""
    dD_H_all = []; dA_W_all = []
    for pid, sub in sess_table.groupby('participant_id'):
        sub = sub.sort_values('createdAt')
        n = len(sub)
        idx = np.arange(1, n + 1, dtype=float)
        d_h = sub['D_H_session'].values
        a_w = sub['A_W_session'].values.copy()
        rng.shuffle(a_w)
        slope_d, intercept_d, *_ = stats.linregress(idx, d_h)
        resid_d = d_h - (intercept_d + slope_d * idx)
        slope_a, intercept_a, *_ = stats.linregress(idx, a_w)
        resid_a = a_w - (intercept_a + slope_a * idx)
        dD_H_all.append(resid_d); dA_W_all.append(resid_a)
    dD_H_all = np.concatenate(dD_H_all); dA_W_all = np.concatenate(dA_W_all)
    rho, _ = stats.spearmanr(dD_H_all, dA_W_all)
    return rho

rng_perm = np.random.default_rng(20260911)
null_rhos_perm = np.array([null_permutation_secondary(session_table_4_eligible, rng_perm) for _ in range(N_REPS_HRS)])
null_rhos_perm = null_rhos_perm[~np.isnan(null_rhos_perm)]
one_sided_p_perm = (null_rhos_perm >= obs_rho_sec).mean()
print('='*100); print('SECONDARY (a) -- within-participant permutation null'); print('='*100)
print(f"N_reps={len(null_rhos_perm)} (target {N_REPS_HRS}), null rho mean={null_rhos_perm.mean():+.4f}, "
      f"SD={null_rhos_perm.std(ddof=1):.4f}")
print(f"One-sided empirical p (fraction of null draws >= observed) = {one_sided_p_perm:.4f}")

SECONDARY (a) -- within-participant permutation null
N_reps=500 (target 500), null rho mean=+0.0075, SD=0.2035
One-sided empirical p (fraction of null draws >= observed) = 0.4720


In [15]:
def null_baseline_mechanical_secondary(sess_table, baseline_df, rng):
    """One Baseline-mechanical pseudo-group draw: for each eligible real participant, resample
    that many Baseline sessions (with replacement) to build one pseudo-participant, using the
    resample-draw order as its session index; compute D_B_session/A_W,B_session per drawn
    session with the same n=1 calibration, then apply the same order-detrending. Returns the
    pooled Spearman(dD_B, dA_W,B) across all pseudo-participants for this one draw."""
    calib_n1 = get_calibration(1, rng)
    baseline_sids_all = baseline_df['session_id'].unique()
    dD_B_all = []; dA_WB_all = []
    for pid, n_sess in sess_table.groupby('participant_id')['session_id'].nunique().items():
        sampled_sids = rng.choice(baseline_sids_all, size=n_sess, replace=True)
        d_b = np.empty(n_sess); a_wb = np.empty(n_sess)
        for i, sid in enumerate(sampled_sids):
            g = baseline_df[baseline_df['session_id'] == sid]
            d_b[i] = (g['hurst_subj_rough'] - g['hurst_demon_rough']).mean()
            z_ws = []
            valid = True
            for W in W_VALUES_HRS:
                tagged = windowed_r_tagged(g, W)
                if len(tagged) < 2:
                    valid = False
                    break
                obs_var = tagged['r'].values.var(ddof=1)
                mean_b_, sd_b_ = calib_n1[W]
                if sd_b_ == 0 or np.isnan(sd_b_):
                    valid = False
                    break
                z_ws.append((obs_var - mean_b_) / sd_b_)
            a_wb[i] = np.mean([max(0.0, -z) for z in z_ws]) if valid else np.nan
        if np.isnan(a_wb).any():
            continue
        idx = np.arange(1, n_sess + 1, dtype=float)
        slope_d, intercept_d, *_ = stats.linregress(idx, d_b)
        resid_d = d_b - (intercept_d + slope_d * idx)
        slope_a, intercept_a, *_ = stats.linregress(idx, a_wb)
        resid_a = a_wb - (intercept_a + slope_a * idx)
        dD_B_all.append(resid_d); dA_WB_all.append(resid_a)
    if len(dD_B_all) < 3:
        return np.nan
    dD_B_all = np.concatenate(dD_B_all); dA_WB_all = np.concatenate(dA_WB_all)
    rho, _ = stats.spearmanr(dD_B_all, dA_WB_all)
    return rho

rng_bmech = np.random.default_rng(20260912)
null_rhos_bmech = np.array([null_baseline_mechanical_secondary(session_table_4_eligible, baseline_4, rng_bmech) for _ in range(N_REPS_HRS)])
null_rhos_bmech = null_rhos_bmech[~np.isnan(null_rhos_bmech)]
one_sided_p_bmech = (null_rhos_bmech >= obs_rho_sec).mean()
print('='*100); print('SECONDARY (b) -- Baseline-mechanical null'); print('='*100)
print(f"N_reps={len(null_rhos_bmech)} (target {N_REPS_HRS}), null rho mean={null_rhos_bmech.mean():+.4f}, "
      f"SD={null_rhos_bmech.std(ddof=1):.4f}")
print(f"One-sided empirical p (fraction of null draws >= observed) = {one_sided_p_bmech:.4f}")

SECONDARY (b) -- Baseline-mechanical null
N_reps=500 (target 500), null rho mean=+0.0836, SD=0.1902
One-sided empirical p (fraction of null draws >= observed) = 0.6240


## Secondary test summary — reading this once run

Report the observed Spearman rho (and secondary Pearson r) together with both one-sided
empirical p-values above, read against the same four interpretive outcome patterns as the
primary test (Hypotheses, A-D), at the within-participant level:

- Pattern A: both nulls' p are small. Real, human-specific within-participant coupling.
- Pattern C: (a)'s p is small but (b)'s is not. Real pairing, but explainable by shared
  computational structure alone, not human-specific.
- Pattern D: neither p is small. No clear within-participant association.

Power caveat: this test's power depends on there being real within-participant, session-to-session
variability in D_H_session/A_W_session. A null result here cannot distinguish "no coupling" from
"no variability to couple" -- see the OSF registration's Statistical models field. This result
does not redefine or get rescued by the primary (between-participant) test's own result above;
both are reported on their own terms.

## Exploratory follow-up (added post-hoc, not part of the registered confirmatory tests)

Motivated by a question raised after seeing the secondary test's N=6 result: for just the two
non-PI Human5+ practitioners (5 eligible sessions each, per `Exp4_Notebook3_Settling_Trajectory_
Analysis.ipynb`'s convention: the PI is the 5+-session participant with the most sessions, 18 in
this dataset), do D_H_session and A_W_session move together at all across their own sessions?

This is descriptive only. No null distribution, no p-value, no claim of significance -- it is
reported as new exploratory work per the Analysis Plan's own rule, not folded into either
confirmatory test above.

In [16]:
PI_ID = human_all_4.groupby('participant_id')['session_id'].nunique().idxmax()  # most sessions = PI, per Notebook3 convention
practitioner_table = session_table_4[
    (session_table_4['participant_id'].isin(human5_4['participant_id'].unique())) &
    (session_table_4['participant_id'] != PI_ID)
].copy()

print(f"PI excluded: {PI_ID}")
print(f"Non-PI Human5+ practitioners: {practitioner_table['participant_id'].nunique()}, "
      f"{len(practitioner_table)} sessions with a defined A_W_session\n")

for pid, sub in practitioner_table.groupby('participant_id'):
    sub = sub.sort_values('createdAt').reset_index(drop=True)
    sub.index = sub.index + 1
    print(f"--- participant {pid} ({len(sub)} sessions) ---")
    print(sub[['D_H_session', 'A_W_session']].to_string())
    print()

raw_rho, raw_p = stats.spearmanr(practitioner_table['D_H_session'], practitioner_table['A_W_session'])
raw_pearson, _ = stats.pearsonr(practitioner_table['D_H_session'], practitioner_table['A_W_session'])
print(f"Raw (session-level, not detrended/centered) Spearman rho = {raw_rho:+.4f} (2-sided p={raw_p:.4f}), "
      f"Pearson r = {raw_pearson:+.4f}")

# Same order-detrending/centering recipe as the confirmatory secondary test, for comparability
practitioner_table_dt = practitioner_table.groupby('participant_id', group_keys=False).apply(detrend_group)
dt_rho, dt_p = stats.spearmanr(practitioner_table_dt['dD_H'], practitioner_table_dt['dA_W'])
print(f"Order-detrended, person-mean-centered Spearman rho = {dt_rho:+.4f} (2-sided p={dt_p:.4f}, "
      f"uninterpretable at N={practitioner_table_dt['participant_id'].nunique()} participants -- descriptive only)")

PI excluded: y612YV0ACNSi8gs6TnC3QIueQi23
Non-PI Human5+ practitioners: 2, 10 sessions with a defined A_W_session

--- participant GkDR0s3OA6eW6Maa6bFVoy37y6b2 (5 sessions) ---
   D_H_session  A_W_session
1     0.019818     0.383024
2    -0.009438     0.294576
3     0.016473     0.870557
4     0.004128     1.117399
5     0.015684     1.290949

--- participant aj2PSAxOmadGQylCj6000WszNWv2 (5 sessions) ---
   D_H_session  A_W_session
1    -0.003596     1.355181
2     0.025331     0.389665
3    -0.011184     0.674416
4     0.033480     0.243195
5    -0.003873     0.513488

Raw (session-level, not detrended/centered) Spearman rho = -0.2970 (2-sided p=0.4047), Pearson r = -0.2435
Order-detrended, person-mean-centered Spearman rho = -0.2485 (2-sided p=0.4888, uninterpretable at N=2 participants -- descriptive only)
